# Phase 2 — Bronze Ingestion
## Ingest raw COMPAS CSV into a Bronze staging table (CSV/Parquet)

This notebook demonstrates a simple ingestion step that copies the raw CSV
into a reproducible Bronze artifact. The notebook uses `pandas` for a
lightweight, local ingestion so it works in a minimal environment.


## 1. Imports & Setup

In [ ]:
import pandas as pd
from pathlib import Path

# Paths (notebook lives in `notebooks/` — resolve relative to repo root)
RAW_PATH = Path('../data/raw/compas-scores-two-years-violent.csv')
BRONZE_DIR = Path('../data/bronze')
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raw path: {RAW_PATH.resolve()}')
print(f'Bronze dir: {BRONZE_DIR.resolve()}')


## 2. Sanity-check the raw CSV

In [ ]:
assert RAW_PATH.exists(), f"Raw file not found at {RAW_PATH}."")



In [ ]:
# Read a small sample to confirm file format
sample = pd.read_csv(RAW_PATH, nrows=5)
print('Sample rows (first 5):')
display(sample)


## 3. Load full CSV (stream-safe) and inspect counts

In [ ]:
# Read full CSV — uses pandas for portability in this repo
df = pd.read_csv(RAW_PATH)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


## 4. Basic validation checks

In [ ]:
# Quick column checks and dtypes
print(df.dtypes.to_string())
print('
Null counts:')
print(df.isnull().sum().sort_values(ascending=False).head(10))


## 5. Persist Bronze artifact (Parquet)

In [ ]:
# Write a Bronze Parquet file for downstream phases
bronze_path = BRONZE_DIR / 'compas_bronze.parquet'
df.to_parquet(bronze_path, index=False)
print(f'Wrote Bronze file to: {bronze_path.resolve()}')


## 6. Verify Bronze artifact contents

In [ ]:
# Read back the parquet to confirm
df_bronze = pd.read_parquet(bronze_path)
print(f'Bronze readback shape: {df_bronze.shape[0]:,} rows × {df_bronze.shape[1]} columns')
df_bronze.head()


## 6a. Data provenance

Record provenance so downstream users know the source and creation step.


In [ ]:
# Provenance details
print('Source file:', RAW_PATH)
print('Bronze artifact:', bronze_path)
print('Rows in raw:', df.shape[0])
print('Rows in bronze:', df_bronze.shape[0])


## 6b. Quick file checks

In [ ]:
# Check bronze file size and existence
import os
print('Bronze file exists:', bronze_path.exists())
print('Bronze file size (bytes):', os.path.getsize(bronze_path))


## 7. Summary & Next Steps

- Bronze artifact created at `data/bronze/compas_bronze.parquet`.
- Next: build Silver pipeline (cleaning, canonical column types).
